# Interior Mutability

In [2]:
use std::collections::HashMap;

struct DnsService {
    lookup_table: HashMap<String, String>
}

impl DnsService {
    fn lookup(&self, hostname: &str) -> Option<&str> {
        self.lookup_table.get(hostname).map(|s| s.as_str())
    }
}

In [3]:
let dns_service = DnsService {
    lookup_table: [("www.example.com".to_string(), "143.33.44.4".to_string()),
            ("www.rust-lang.org".to_string(), "1.22.33.1".to_string()),
            ("www.mozilla.org".to_string(), "2.43.44.3".to_string())
        ].iter().cloned().collect()
};

println!("{:?}", dns_service.lookup("www.rust-lang.org"));

Some("1.22.33.1")


In [4]:
use std::collections::HashMap;

struct DnsService {
    lookup_table: HashMap<String, String>,
    counter: i32
}

impl DnsService {
    fn lookup(&mut self, hostname: &str) -> Option<&str> {
        self.counter += 1; // Error: cannot assign to `self.counter` because it is borrowed
        self.lookup_table.get(hostname).map(|s| s.as_str())
    }
}

fn make_requests(dns_service: &mut DnsService) {
    let hostnames = ["www.example.com", "www.rust-lang.org", "www.mozilla.org"];
    for hostname in hostnames.iter() {
        println!("{:?}", dns_service.lookup(hostname));
    }
}

let mut dns_service = DnsService {
    lookup_table: [("www.example.com".to_string(), "143.33.44.4".to_string()),
            ("www.rust-lang.org".to_string(), "1.22.33.1".to_string()),
            ("www.mozilla.org".to_string(), "2.43.44.3".to_string())
        ].iter().cloned().collect(),
        counter: 0
};

make_requests(&mut dns_service);

The type of the variable dns_service was redefined, so was lost.


Some("143.33.44.4")
Some("1.22.33.1")
Some("2.43.44.3")


## `Cell` - for `Copy` types

In [5]:
use std::collections::HashMap;
use std::cell::Cell;

struct DnsService {
    lookup_table: HashMap<String, String>,
    counter: Cell<i32>
}

impl DnsService {
    fn lookup(&self, hostname: &str) -> Option<&str> {
        let value = self.counter.get();
        self.counter.set(value + 1);
        self.lookup_table.get(hostname).map(|s| s.as_str())
    }
}

In [6]:
fn main() {
    let dns_service = DnsService {
        lookup_table: [("www.example.com".to_string(), "143.33.44.4".to_string()),
                ("www.rust-lang.org".to_string(), "1.22.33.1".to_string()),
                ("www.mozilla.org".to_string(), "2.43.44.3".to_string())
            ].iter().cloned().collect(), 
            counter: Cell::new(0)
    };


    let ref_dns_service = &dns_service;

    println!("{:?}", dns_service.lookup("www.rust-lang.org"));
    println!("{:?}", ref_dns_service.lookup("www.example.com"));
    println!("{:?}", dns_service.lookup("www.infotraining.pl"));

    fn make_requests(dns_service: &DnsService) {
        let hostnames = ["www.example.com", "www.rust-lang.org", "www.mozilla.org"];
        for hostname in hostnames.iter() {
            println!("{:?}", dns_service.lookup(hostname));
        }
    }

    make_requests(&dns_service);

    println!("Dns used {} times", dns_service.counter.get());
}

main();

Some("1.22.33.1")
Some("143.33.44.4")
None
Some("143.33.44.4")
Some("1.22.33.1")
Some("2.43.44.3")
Dns used 6 times


## RefCell - for non-`Copy` types

In [16]:
use std::collections::HashMap;
use std::cell::RefCell;

type LookupLog = Vec<String>;

struct DnsServiceWithRefCell {
    lookup_table: HashMap<String, String>,
    lookup_log: RefCell<LookupLog>,
}

impl DnsServiceWithRefCell {
    fn lookup(&self, hostname: &str) -> Option<&str> {
        
        {
            let shared_ref_to_log: std::cell::Ref<LookupLog> = self.lookup_log.borrow();
            println!("Shared ref to log: {:?}", shared_ref_to_log);
        } // drop the shared reference to the log here, so we can borrow mutably below
        self.lookup_log.borrow_mut().push(hostname.to_string());

        self.lookup_table.get(hostname).map(|s| s.as_str())
    }
}

In [17]:
let dns_service = DnsServiceWithRefCell {
    lookup_table: [("www.example.com".to_string(), "143.33.44.4".to_string()),
            ("www.rust-lang.org".to_string(), "1.22.33.1".to_string()),
            ("www.mozilla.org".to_string(), "2.43.44.3".to_string())
        ].iter().cloned().collect(), 
    lookup_log: RefCell::new(Vec::new())
};

println!("{:?}", dns_service.lookup("www.rust-lang.org"));
println!("{:?}", dns_service.lookup("www.example.com"));
println!("{:?}", dns_service.lookup("www.infotraining.pl"));

println!("Dns looked up for {:?}", dns_service.lookup_log.borrow());

Shared ref to log: []
Some("1.22.33.1")
Shared ref to log: ["www.rust-lang.org"]
Some("143.33.44.4")
Shared ref to log: ["www.rust-lang.org", "www.example.com"]
None
Dns looked up for ["www.rust-lang.org", "www.example.com", "www.infotraining.pl"]


## RwLock - for thread-safe shared access

In [20]:
use std::collections::HashMap;
use std::cell::RefCell;

type LookupLog = Vec<String>;

struct DnsServiceUnsafe {
    lookup_table: HashMap<String, String>,
    lookup_log: RefCell<LookupLog>,
}

impl DnsServiceUnsafe {
    fn from_slice(data: &[(&str, &str)]) -> Self {
        DnsServiceUnsafe {
            lookup_table: data.iter().cloned().map(|(k, v)| (k.to_string(), v.to_string())).collect(),
            lookup_log: RefCell::new(Vec::new())
        }
    }

    fn lookup(&self, hostname: &str) -> Option<&str> {
        self.lookup_log.borrow_mut().push(hostname.to_string());
        self.lookup_table.get(hostname).map(|s| s.as_str())
    }
}

In [ ]:
fn main() {
    let dns_service = DnsServiceUnsafe::from_slice(&[("www.example.com", "122.33.22.33"), ("www.rust-lang.org", "1.2.3.4"), ("www.mozilla.org", "9.4.3.4")]);

    println!("{:?}", dns_service.lookup("www.rust-lang.org"));

    let thd_1 = std::thread::spawn(|| { // closure borrows shared reference to dns_service: &dns_service
        println!("{:?}", dns_service.lookup("www.example.com"));
    });

    thd_1.join().unwrap();
}

main();

Error: `RefCell<Vec<String>>` cannot be shared between threads safely

## Arc + RwLock - for shared access across threads

In [21]:
use std::collections::HashMap;
use std::sync::RwLock;
use std::borrow::BorrowMut;

type LookupLog = Vec<String>;

struct DnsServiceThreadSafe {
    lookup_table: HashMap<String, String>,
    lookup_log: RwLock<LookupLog>,
}

impl DnsServiceThreadSafe {
    fn from_slice(data: &[(&str, &str)]) -> DnsServiceThreadSafe {
        DnsServiceThreadSafe {
            lookup_table: data.iter().cloned().map(|(k, v)| (k.to_string(), v.to_string())).collect(),
            lookup_log: RwLock::new(Vec::new())
        }
    }

    fn lookup(&self, hostname: &str) -> Option<&str> {
        {
            let mut log: std::sync::RwLockWriteGuard<LookupLog> = self.lookup_log.write().unwrap();
            log.push(hostname.to_string());
        }
        self.lookup_table.get(hostname).map(|s| s.as_str())
    }
}

In [23]:
use std::sync::Arc;

fn main() {
    let dns_service = Arc::new(
        DnsServiceThreadSafe::from_slice(&[("www.example.com", "122.33.22.33"), ("www.rust-lang.org", "1.2.3.4"), ("www.mozilla.org", "9.4.3.4")])
    );

    println!("{:?}", dns_service.lookup("www.rust-lang.org"));

    let mut dns = dns_service.clone();
    let thd_1 = std::thread::spawn(move || {
        println!("{:?}", dns.lookup("www.example.com"));
    });

    let mut dns = dns_service.clone();
    let thd_2 = std::thread::spawn(move || {
        println!("{:?}", dns.lookup("www.mozilla.org"));
    });

    thd_1.join().unwrap();
    thd_2.join().unwrap();

    println!("Dns looked up for {:?}", dns_service.lookup_log.read().unwrap());
}

main();

Some("1.2.3.4")
Some("122.33.22.33")
Some("9.4.3.4")
Dns looked up for ["www.rust-lang.org", "www.example.com", "www.mozilla.org"]
